In [2]:
import warnings
from statsbombpy.api_client import NoAuthWarning
warnings.filterwarnings('ignore', category=NoAuthWarning)

In [3]:
from statsbombpy import sb
import pandas as pd

competitions = sb.competitions()

# StatsBomb open data key competitions — these have the most complete event data
target_comps = [
    (11, 27),   # La Liga 2015/16
    (11, 26),   # La Liga 2014/15  
    (11, 25),   # La Liga 2013/14
    (2, 44),    # Premier League 2003/04
    (72, 30),   # Women's World Cup 2019
    (37, 42),   # FA Women's Super League
]

print(f"Pulling matches for {len(target_comps)} competition-seasons...")

Pulling matches for 6 competition-seasons...


In [4]:
all_matches = []

for comp_id, season_id in target_comps:
    try:
        matches = sb.matches(competition_id=comp_id, season_id=season_id)
        matches['competition_id'] = comp_id
        matches['season_id'] = season_id
        all_matches.append(matches)
        print(f"  comp={comp_id} season={season_id}: {len(matches)} matches")
    except Exception as e:
        print(f"  comp={comp_id} season={season_id}: FAILED — {e}")

all_matches_df = pd.concat(all_matches, ignore_index=True)
print(f"\nTotal matches: {len(all_matches_df)}")

  comp=11 season=27: 380 matches
  comp=11 season=26: 38 matches
  comp=11 season=25: 31 matches
  comp=2 season=44: 38 matches
  comp=72 season=30: 52 matches
  comp=37 season=42: 87 matches

Total matches: 626


In [5]:
all_gk_events = []
all_shot_events = []
all_pass_events = []

match_ids = all_matches_df['match_id'].tolist()
print(f"Processing {len(match_ids)} matches...")

for i, match_id in enumerate(match_ids):
    if i % 20 == 0:
        print(f"  {i}/{len(match_ids)}...")
    try:
        events = sb.events(match_id=match_id)
        
        # GK actions
        gk = events[events['type'] == 'Goal Keeper'].copy()
        gk['match_id'] = match_id
        all_gk_events.append(gk)
        
        # Shots (for PSxG)
        shots = events[events['type'] == 'Shot'].copy()
        shots['match_id'] = match_id
        all_shot_events.append(shots)
        
        # GK passes (for distribution)
        passes = events[
            (events['type'] == 'Pass') & 
            (events['position'] == 'Goalkeeper')
        ].copy()
        passes['match_id'] = match_id
        all_pass_events.append(passes)
        
    except Exception as e:
        print(f"  match {match_id} failed: {e}")

gk_events = pd.concat(all_gk_events, ignore_index=True)
shot_events = pd.concat(all_shot_events, ignore_index=True)
pass_events = pd.concat(all_pass_events, ignore_index=True)

print(f"\nGK events: {gk_events.shape}")
print(f"Shot events: {shot_events.shape}")
print(f"Pass events: {pass_events.shape}")

Processing 626 matches...
  0/626...
  20/626...
  40/626...
  60/626...
  80/626...
  100/626...
  120/626...
  140/626...
  160/626...
  180/626...
  200/626...
  220/626...
  240/626...
  260/626...
  280/626...
  300/626...
  320/626...
  340/626...
  360/626...
  380/626...
  400/626...
  420/626...
  440/626...
  460/626...
  480/626...
  500/626...
  520/626...
  540/626...
  560/626...
  580/626...
  600/626...
  620/626...

GK events: (19054, 121)
Shot events: (15375, 121)
Pass events: (40711, 121)


In [6]:
# Derive GK minutes from events — more reliable than lineup timestamps
all_minutes = []

for match_id in match_ids:
    try:
        events = sb.events(match_id=match_id)
        
        # Get match duration from the last event timestamp
        max_minute = events['minute'].max()
        
        # Find GKs who played in this match
        gk_players = events[events['position'] == 'Goalkeeper'][['player_id', 'player', 'team']].drop_duplicates()
        
        for _, row in gk_players.iterrows():
            # Get first and last minute this GK appears in events
            player_events = events[events['player_id'] == row['player_id']]
            first_min = player_events['minute'].min()
            last_min = player_events['minute'].max()
            
            all_minutes.append({
                'match_id': match_id,
                'player_id': row['player_id'],
                'player_name': row['player'],
                'team': row['team'],
                'minutes_played': last_min - first_min + 1,
                'match_duration': max_minute
            })
    except:
        pass

minutes_df = pd.DataFrame(all_minutes)
print(minutes_df.head(10))
print(f"\nTotal GK-match records: {len(minutes_df)}")
print(f"\nMinutes played distribution:")
print(minutes_df['minutes_played'].describe())

   match_id  player_id                       player_name  \
0   3825848    10763.0              Asier Riesgo Unamuno   
1   3825848     6730.0       Rubén Iván Martínez Andrade   
2   3825895    26753.0              Javier Varas Herrera   
3   3825895     6823.0              Sergio Rico González   
4   3825894    27457.0            Manuel Fernández Muñíz   
5   3825894     6625.0           Vicente Guaita Panadero   
6   3825855    23843.0               Diego Mariño Villar   
7   3825855     5577.0  Francisco Guillermo Ochoa Magaña   
8   3825908     6792.0                  Pau López Sabata   
9   3825908    27462.0  Xabier Iruretagoiena Aranzamendi   

                     team  minutes_played  match_duration  
0                   Eibar              92              93  
1              Levante UD              87              93  
2              Las Palmas              94              93  
3                 Sevilla              89              93  
4  RC Deportivo La Coruña              

In [7]:
# ── Metric 1: Save % and PSxG-GA ──────────────────────────────────────────

# First, get goals conceded and saves per GK per match
saves = gk_events[gk_events['goalkeeper_type'] == 'Shot Saved'].groupby(
    ['match_id', 'player', 'player_id']
).size().reset_index(name='saves')

goals_conceded = gk_events[gk_events['goalkeeper_type'] == 'Goal Conceded'].groupby(
    ['match_id', 'player', 'player_id']
).size().reset_index(name='goals_conceded')

# PSxG: sum of xG on shots faced by each GK
# Link via match_id and team
shot_xg = shot_events[shot_events['shot_outcome'].isin(['Goal', 'Saved', 'Saved To Post'])].copy()
shot_xg = shot_events[shot_events['shot_outcome'].isin(['Goal', 'Saved'])].copy()

# Get each GK's team per match from minutes_df
gk_teams = minutes_df[['match_id', 'player_id', 'player_name', 'team', 'minutes_played']].copy()

# For each shot, the GK who faced it is on the OPPOSING team in that match
# Join shots to GKs where shot team != gk team (same match)
shot_xg_slim = shot_xg[['match_id', 'team', 'shot_statsbomb_xg']].copy()
shot_xg_slim.columns = ['match_id', 'shooting_team', 'xg']

gk_psxg = gk_teams.merge(shot_xg_slim, on='match_id')
gk_psxg = gk_psxg[gk_psxg['team'] != gk_psxg['shooting_team']]
gk_psxg = gk_psxg.groupby(['match_id', 'player_id', 'player_name', 'team', 'minutes_played'])['xg'].sum().reset_index()
gk_psxg.columns = ['match_id', 'player_id', 'player_name', 'team', 'minutes_played', 'psxg']

print("PSxG sample:")
print(gk_psxg.head())

PSxG sample:
   match_id  player_id       player_name                    team  \
0     22921    25449.0     Min-Jeong Kim  Korea Republic Women's   
1     22924    10394.0  Ingrid Hjelmseth          Norway Women's   
2     22924    25467.0   Tochukwu Oluehi         Nigeria Women's   
3     22926    10257.0     Almuth Schult         Germany Women's   
4     22926    25483.0      Shimeng Peng        China PR Women's   

   minutes_played      psxg  
0              91  0.678634  
1              96  0.085538  
2              87  0.411832  
3              86  0.149528  
4              78  0.224641  


In [8]:
# ── Metric 2: Sweeper actions ─────────────────────────────────────────────
sweeper = gk_events[gk_events['goalkeeper_type'] == 'Keeper Sweeper'].groupby(
    ['match_id', 'player', 'player_id']
).size().reset_index(name='sweeper_actions')

# ── Metric 3: Claiming (Collected + Punch) ────────────────────────────────
claiming = gk_events[gk_events['goalkeeper_type'].isin(['Collected', 'Punch'])].groupby(
    ['match_id', 'player', 'player_id']
).size().reset_index(name='claiming_actions')

# ── Metric 4: Distribution (passes) ──────────────────────────────────────
pass_success = pass_events.copy()
pass_success['completed'] = pass_events['pass_outcome'].isna()  # NaN outcome = completed in StatsBomb
pass_success['is_long'] = pass_events['pass_length'] > 32  # ~35 yards threshold

distribution = pass_success.groupby(['match_id', 'player', 'player_id']).agg(
    total_passes=('completed', 'count'),
    completed_passes=('completed', 'sum'),
    long_balls=('is_long', 'sum')
).reset_index()
distribution['pass_completion_pct'] = distribution['completed_passes'] / distribution['total_passes'] * 100
distribution['long_ball_pct'] = distribution['long_balls'] / distribution['total_passes'] * 100

print("Distribution sample:")
print(distribution.head())

Distribution sample:
   match_id            player  player_id  total_passes  completed_passes  \
0     22921     Min-Jeong Kim    25449.0            40                23   
1     22921    Sarah Bouhaddi    10129.0            17                17   
2     22924  Ingrid Hjelmseth    10394.0            39                27   
3     22924   Tochukwu Oluehi    25467.0            18                10   
4     22926     Almuth Schult    10257.0            38                19   

   long_balls  pass_completion_pct  long_ball_pct  
0          21            57.500000      52.500000  
1           0           100.000000       0.000000  
2          21            69.230769      53.846154  
3          11            55.555556      61.111111  
4          28            50.000000      73.684211  


In [9]:
# Drop redundant 'player' name column — we keep player_name from gk_psxg
saves_clean = saves.drop(columns=['player'])
goals_clean = goals_conceded.drop(columns=['player'])
sweeper_clean = sweeper.drop(columns=['player'])
claiming_clean = claiming.drop(columns=['player'])

# ── Master merge ──────────────────────────────────────────────────────────
master = gk_psxg.copy()

master = master.merge(saves_clean, on=['match_id', 'player_id'], how='left')
master = master.merge(goals_clean, on=['match_id', 'player_id'], how='left')
master = master.merge(sweeper_clean, on=['match_id', 'player_id'], how='left')
master = master.merge(claiming_clean, on=['match_id', 'player_id'], how='left')
master = master.merge(
    distribution[['match_id', 'player_id', 'total_passes', 'pass_completion_pct', 'long_ball_pct']],
    on=['match_id', 'player_id'], how='left'
)

# Fill NaN counts with 0
count_cols = ['saves', 'goals_conceded', 'sweeper_actions', 'claiming_actions', 'total_passes']
master[count_cols] = master[count_cols].fillna(0)

# ── Per-90 normalisation ──────────────────────────────────────────────────
master['saves_p90'] = master['saves'] / master['minutes_played'] * 90
master['sweeper_p90'] = master['sweeper_actions'] / master['minutes_played'] * 90
master['claiming_p90'] = master['claiming_actions'] / master['minutes_played'] * 90
master['psxg_p90'] = master['psxg'] / master['minutes_played'] * 90

# ── Save percentage ───────────────────────────────────────────────────────
master['shots_faced'] = master['saves'] + master['goals_conceded']
master['save_pct'] = master['saves'] / master['shots_faced'].replace(0, float('nan')) * 100

# ── PSxG-GA (positive = outperforming expectations) ──────────────────────
master['psxg_ga'] = master['psxg'] - master['goals_conceded']

print(f"Master table shape: {master.shape}")
print(master[['player_name', 'team', 'minutes_played', 'saves', 'goals_conceded',
              'psxg', 'psxg_ga', 'save_pct', 'sweeper_p90']].head(10))

Master table shape: (1239, 20)
                    player_name                    team  minutes_played  \
0                 Min-Jeong Kim  Korea Republic Women's              91   
1              Ingrid Hjelmseth          Norway Women's              96   
2               Tochukwu Oluehi         Nigeria Women's              87   
3                 Almuth Schult         Germany Women's              86   
4                  Shimeng Peng        China PR Women's              78   
5  Sandra Paños García-Villamil           Spain Women's              95   
6                Andile Dlamini    South Africa Women's              91   
7  Lydia Grace Yilkari Williams       Australia Women's              91   
8                Laura Giuliani           Italy Women's              91   
9              Sydney Schneider         Jamaica Women's              93   

   saves  goals_conceded      psxg   psxg_ga    save_pct  sweeper_p90  
0    4.0             4.0  0.678634 -3.321366   50.000000     0.000000  

In [10]:
# ── Aggregate per-player across all matches ───────────────────────────────
player_agg = master.groupby(['player_id', 'player_name']).agg(
    total_minutes=('minutes_played', 'sum'),
    total_saves=('saves', 'sum'),
    total_goals_conceded=('goals_conceded', 'sum'),
    total_psxg=('psxg', 'sum'),
    total_sweeper=('sweeper_actions', 'sum'),
    total_claiming=('claiming_actions', 'sum'),
    total_passes=('total_passes', 'sum'),
    avg_pass_completion=('pass_completion_pct', 'mean'),
    avg_long_ball_pct=('long_ball_pct', 'mean'),
    matches_played=('match_id', 'count')
).reset_index()

# Per-90 at player level
player_agg['saves_p90'] = player_agg['total_saves'] / player_agg['total_minutes'] * 90
player_agg['sweeper_p90'] = player_agg['total_sweeper'] / player_agg['total_minutes'] * 90
player_agg['claiming_p90'] = player_agg['total_claiming'] / player_agg['total_minutes'] * 90
player_agg['shots_faced'] = player_agg['total_saves'] + player_agg['total_goals_conceded']
player_agg['save_pct'] = player_agg['total_saves'] / player_agg['shots_faced'].replace(0, float('nan')) * 100
player_agg['psxg_ga'] = player_agg['total_psxg'] - player_agg['total_goals_conceded']

# Filter: minimum 450 minutes (5 full matches) to remove noise
player_agg_filtered = player_agg[player_agg['total_minutes'] >= 450].copy()

print(f"GKs with 450+ minutes: {len(player_agg_filtered)}")
print(player_agg_filtered.sort_values('psxg_ga', ascending=False).head(10)[
    ['player_name', 'total_minutes', 'save_pct', 'psxg_ga', 'sweeper_p90']
])

GKs with 450+ minutes: 55
                    player_name  total_minutes   save_pct   psxg_ga  \
5                 Ellie Roebuck           1395  86.792453  0.822234   
40               Laura Giuliani            450  85.714286  0.520973   
44             Ingrid Hjelmseth            497  70.000000  0.334748   
46          Sari van Veenendaal            656  81.818182  0.330567   
100  Antonio Rodríguez Martínez            909  70.000000  0.272592   
42                Almuth Schult            458  83.333333 -0.373362   
9         Alyssa Michele Naeher            543  80.000000 -0.981366   
114        Mary Alexandra Earps           1282  79.545455 -1.867255   
6            Rut Hedvig Lindahl            663  75.000000 -1.889375   
0                   Mathew Ryan            723  73.333333 -3.094613   

     sweeper_p90  
5       0.322581  
40      0.800000  
44      0.181087  
46      0.274390  
100     1.782178  
42      0.589520  
9       0.497238  
114     0.421217  
6       1.221719  
0 

In [15]:
import duckdb
import os

os.makedirs('../data', exist_ok=True)

con = duckdb.connect('../data/gk_scout.duckdb')

# Save both tables
con.execute("DROP TABLE IF EXISTS gk_match_level")
con.execute("DROP TABLE IF EXISTS gk_player_level")

con.execute("CREATE TABLE gk_match_level AS SELECT * FROM master")
con.execute("CREATE TABLE gk_player_level AS SELECT * FROM player_agg_filtered")

# Verify
print("Tables saved:")
print(con.execute("SELECT COUNT(*) FROM gk_match_level").fetchone())
print(con.execute("SELECT COUNT(*) FROM gk_player_level").fetchone())

# Add competition labels — useful for dashboard filters
con.execute("DROP TABLE IF EXISTS competitions_used")
con.execute("""
    CREATE TABLE competitions_used AS
    SELECT * FROM (VALUES
        (11, 27, 'La Liga 2015/16'),
        (11, 26, 'La Liga 2014/15'),
        (11, 25, 'La Liga 2013/14'),
        (2,  44, 'Premier League 2003/04'),
        (72, 30, 'Women''s World Cup 2019'),
        (37, 42, 'FA WSL')
    ) t(competition_id, season_id, competition_name)
""")

con.close()
print("\nDuckDB saved to ../data/gk_scout.duckdb")

Tables saved:
(1239,)
(55,)

DuckDB saved to ../data/gk_scout.duckdb


In [11]:
# Add competition labels to match-level data
comp_map = {
    (11, 27): 'La Liga 2015/16',
    (11, 26): 'La Liga 2014/15',
    (11, 25): 'La Liga 2013/14',
    (2,  44): 'Premier League 2003/04',
    (72, 30): "Women's World Cup 2019",
    (37, 42): 'FA WSL',
}

# Add comp/season back to matches df then merge into master
match_comp = all_matches_df[['match_id', 'competition_id', 'season_id']].copy()
match_comp['competition'] = match_comp.apply(
    lambda r: comp_map.get((r['competition_id'], r['season_id']), 'Unknown'), axis=1
)

master_with_comp = master.merge(match_comp[['match_id', 'competition']], on='match_id', how='left')

# Verify
print(master_with_comp['competition'].value_counts())
print(master_with_comp[['player_name', 'competition']].head(10))

competition
La Liga 2015/16           756
FA WSL                    172
Women's World Cup 2019     99
Premier League 2003/04     76
La Liga 2014/15            72
La Liga 2013/14            64
Name: count, dtype: int64
                    player_name             competition
0                 Min-Jeong Kim  Women's World Cup 2019
1              Ingrid Hjelmseth  Women's World Cup 2019
2               Tochukwu Oluehi  Women's World Cup 2019
3                 Almuth Schult  Women's World Cup 2019
4                  Shimeng Peng  Women's World Cup 2019
5  Sandra Paños García-Villamil  Women's World Cup 2019
6                Andile Dlamini  Women's World Cup 2019
7  Lydia Grace Yilkari Williams  Women's World Cup 2019
8                Laura Giuliani  Women's World Cup 2019
9              Sydney Schneider  Women's World Cup 2019


In [12]:
# For each player, assign the competition they played most minutes in
player_comp = master_with_comp.groupby(
    ['player_id', 'player_name', 'competition']
)['minutes_played'].sum().reset_index()

player_comp = player_comp.sort_values('minutes_played', ascending=False)
player_primary_comp = player_comp.drop_duplicates(subset='player_id')[['player_id', 'competition']]
player_primary_comp.columns = ['player_id', 'primary_competition']

# Merge into player_agg_filtered
player_agg_filtered_v2 = player_agg_filtered.merge(player_primary_comp, on='player_id', how='left')

print(player_agg_filtered_v2[['player_name', 'primary_competition', 'total_minutes']].head(10))
print(f"\nCompetition breakdown:")
print(player_agg_filtered_v2['primary_competition'].value_counts())

                        player_name     primary_competition  total_minutes
0                       Mathew Ryan         La Liga 2015/16            723
1                   Alphonse Areola         La Liga 2015/16           2841
2                     Ellie Roebuck                  FA WSL           1395
3                Rut Hedvig Lindahl  Women's World Cup 2019            663
4        Claudio Andrés Bravo Muñoz         La Liga 2014/15           5937
5             Alyssa Michele Naeher  Women's World Cup 2019            543
6  Francisco Guillermo Ochoa Magaña         La Liga 2015/16            939
7               Keylor Navas Gamboa         La Liga 2015/16           3105
8                         Jan Oblak         La Liga 2015/16           3146
9              Gorka Iraizoz Moreno         La Liga 2015/16           3451

Competition breakdown:
primary_competition
La Liga 2015/16           33
FA WSL                    12
Women's World Cup 2019     6
La Liga 2013/14            2
La Liga 2014/15

In [ ]:
# import duckdb
# con = duckdb.connect('../data/gk_scout.duckdb')

In [16]:
con = duckdb.connect('../data/gk_scout.duckdb')

con.execute("DROP TABLE IF EXISTS gk_match_level")
con.execute("DROP TABLE IF EXISTS gk_player_level")

con.execute("CREATE TABLE gk_match_level AS SELECT * FROM master_with_comp")
con.execute("CREATE TABLE gk_player_level AS SELECT * FROM player_agg_filtered_v2")

print("Updated tables:")
print(con.execute("SELECT COUNT(*) FROM gk_match_level").fetchone())
print(con.execute("SELECT COUNT(*) FROM gk_player_level").fetchone())
print(con.execute("SELECT primary_competition, COUNT(*) FROM gk_player_level GROUP BY primary_competition").df())

con.close()

Updated tables:
(1239,)
(55,)
      primary_competition  count_star()
0         La Liga 2014/15             1
1  Premier League 2003/04             1
2         La Liga 2013/14             2
3                  FA WSL            12
4  Women's World Cup 2019             6
5         La Liga 2015/16            33
